# Sesión 2 · Ejercicio 1 — El juego
**Objetivo:** entrenar una GAN, compararla contra el VAE de ayer sobre el
mismo conjunto, y romperla a propósito midiendo la cobertura de modos.
**Tiempo:** MÍNIMO 25 min · COMPLETO 45 min
**Produce:** §3 de su bitácora — comparación + **los dos números de cobertura**
**Necesitas:** el setup activo (el VAE de ayer se recupera solo)


### Cómo trabajar este cuaderno (1 minuto de lectura)

1. **Guarde su copia**: Archivo → Guardar una copia en Drive. Si no, pierde su trabajo al cerrar.
2. Ejecute las celdas **en orden**. Solo las marcadas `#### OBLIGATORIO ####` producen su entregable; las de **EXTENSIÓN** son opcionales, para quien le sobre tiempo.
3. ¿Algo no corre, o tarda demasiado? Ejecute la **CELDA DE RESCATE**: carga resultados ya calculados y usted sigue con el análisis. Usarla **no descuenta puntos** — solo dígalo en su bitácora.
4. Al terminar, copie la figura y sus observaciones (2-3 líneas con sus palabras) a la sección de su **bitácora** que dice el encabezado. Eso es TODO el entregable — no se pide nada más.


In [ ]:
#### OBLIGATORIO #### — setup (idempotente: puede ejecutarla dos veces)
import os, sys
if not os.path.isdir("src"):
    if not os.path.isdir("IAA6_M13_Gen"):
        !git clone -q https://github.com/AdriannaGmz/IAA6_M13_Gen
    %cd IAA6_M13_Gen
!pip install -q -r requirements.txt
sys.path.insert(0, ".")
from src import datos, modelos, evaluar, graficas, rescate
MODO_GPU = rescate.hay_gpu()   # imprime "GPU disponible" o "Modo CPU"


In [ ]:
#### OBLIGATORIO #### — el conjunto común y el VAE de ayer
X, y, meta = datos.cargar("imagen")

# No hace falta reentrenar el VAE: se recupera del rescate de ayer.
contenido, _ = rescate.cargar("s1_vae")
vae = modelos.VAE(meta, dim_latente=32)
vae.codificador.load_state_dict(contenido["state_dicts"]["codificador"])
vae.decodificador.load_state_dict(contenido["state_dicts"]["decodificador"])
print("VAE de ayer recuperado.")


In [ ]:
#### OBLIGATORIO #### — entrenar la GAN equilibrada (~1-2 min en CPU)
gan = modelos.GAN(meta, dim_ruido=64)
historial = gan.entrenar(
    X, epocas=15,
    cb=lambda e, r: print(f"  época {e + 1:2d}/15 · "
                          f"D {r['perdida_d']:.2f} · G {r['perdida_g']:.2f}"))
graficas.curva(historial, "El juego: dos pérdidas en tensión")


In [ ]:
#### OBLIGATORIO #### — cara a cara: el VAE de ayer vs la GAN de hoy
fig = graficas.comparar(vae.muestrear(16), gan.muestrear(16),
                        etiquetas=("VAE (ayer)", "GAN (hoy)"))
fig.savefig("bitacora_s2e1_vae_vs_gan.png", dpi=120)


### Observación 1 — dos renglones

- ¿Cuál de los dos produce **bordes más nítidos**? ___
- ¿Cuál produce **figuras más reconocibles**? ___

(No siempre gana el mismo en las dos preguntas. Eso también es un dato.)


In [ ]:
#### OBLIGATORIO #### — medir la cobertura de modos de cada uno
# COMPLETAR: mida qué fracción de los 5 modos del conjunto cubre
# cada modelo.
# Pista: la función está en src/evaluar, recibe (muestras_sinteticas,
#        X_reales, k=5) y devuelve un número entre 0 y 1.
#        Use vae.muestrear(500) y gan.muestrear(500).
cob_vae = ...
cob_gan = ...
print(f"cobertura VAE: {cob_vae:.2f}   cobertura GAN: {cob_gan:.2f}")


In [ ]:
#### OBLIGATORIO #### — romper la GAN a propósito
# COMPLETAR: entrene una GAN nueva 6 épocas dándole al discriminador
# una tasa de aprendizaje 10 veces mayor que la del generador.
# Pista: use lr_g = 1e-4. Entonces, ¿cuánto vale lr_d?
#        gan_rota.entrenar(X, epocas=6, lr_d=..., lr_g=...)
gan_rota = modelos.GAN(meta, dim_ruido=64)
gan_rota.entrenar(X, epocas=6, lr_d=..., lr_g=...)
fig = graficas.rejilla(gan_rota.muestrear(16), "GAN con lr_d = 10·lr_g")
fig.savefig("bitacora_s2e1_gan_rota.png", dpi=120)


In [ ]:
#### OBLIGATORIO #### — el número que delata el colapso
# COMPLETAR: mida la cobertura de modos de la GAN rota, igual que
# arriba, y compare los dos números.
# Pista: misma función, ahora con gan_rota.muestrear(500).
cob_rota = ...
print(f"cobertura GAN equilibrada: {cob_gan:.2f}")
print(f"cobertura GAN rota:        {cob_rota:.2f}")


In [ ]:
# ── CELDA DE RESCATE ────────────────────────────────────────
# ¿No entrenó alguna de las dos GAN? Ejecute esto y siga.
c_eq, _ = rescate.cargar("s2_gan")
gan = modelos.GAN(meta, dim_ruido=64)
gan.generador.load_state_dict(c_eq["state_dicts"]["generador"])
gan.discriminador.load_state_dict(c_eq["state_dicts"]["discriminador"])
c_ro, _ = rescate.cargar("s2_gan_colapso")
gan_rota = modelos.GAN(meta, dim_ruido=64)
gan_rota.generador.load_state_dict(c_ro["state_dicts"]["generador"])
gan_rota.discriminador.load_state_dict(c_ro["state_dicts"]["discriminador"])
cob_gan, cob_rota = c_eq["cobertura"], c_ro["cobertura"]
print(f"Recuperado. cobertura equilibrada {cob_gan:.2f} · "
      f"rota {cob_rota:.2f}")


### Observación 2 — ⚠️ los dos números van a la bitácora

- **Cobertura equilibrada:** ___
- **Cobertura tras desbalancear (lr_d = 10·lr_g):** ___
- Las 16 muestras de la GAN rota: ¿cuántas figuras DISTINTAS ve? ___

⚠️ **Guarde los dos números en §3. Se usan de nuevo en la Sesión 4**,
cuando un número parecido va a decidir si un conjunto sintético sirve.


In [ ]:
#### OBLIGATORIO #### — artefacto para la bitácora
print("Copie este bloque en la sección §3 de su bitácora y adjunte")
print("bitacora_s2e1_vae_vs_gan.png y bitacora_s2e1_gan_rota.png:\n")
print(f"- Cobertura de modos VAE: {cob_vae:.2f}")
print(f"- Cobertura GAN equilibrada: {cob_gan:.2f}")
print(f"- Cobertura GAN rota (lr_d = 10·lr_g): {cob_rota:.2f}")
print("- Bordes más nítidos: <VAE/GAN>  · Más figuras distintas: <...>")


### EXTENSIÓN (equipos rápidos)
¿Dónde está la frontera del colapso? Pruebe razones intermedias
(lr_d = 2·lr_g, 5·lr_g) con pocas épocas y anote la cobertura de cada
una. ¿El colapso llega gradual o de golpe?


In [ ]:
#### EXTENSIÓN ####
for razon in (2, 5):
    g = modelos.GAN(meta, dim_ruido=64)
    g.entrenar(X, epocas=6, lr_d=razon * 1e-4, lr_g=1e-4)
    c = evaluar.cobertura_modos(g.muestrear(500), X, k=5)
    print(f"lr_d = {razon}·lr_g → cobertura {c:.2f}")
